# RoadGuard
#### A pothole detection system

This notebook will walk through the development of our ML classification model that identifies potholes on roads.

## Setup

Import required packages:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


## Data Preprocessing

In [3]:
temp_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224), #final input dimension
    transforms.ToTensor()
])

temp_dataset = datasets.ImageFolder('./data', transform=temp_transform)
temp_loader = DataLoader(temp_dataset, batch_size=64, shuffle=False, num_workers=4)

In [4]:
def calculate_mean(loader):
    channel_sum = torch.zeros(3)
    channel_count = 0

    for images, _ in loader:
        channel_sum += torch.mean(images, dim=[0, 2, 3]) * images.shape[0] #means of RGB channels per batch
        channel_count += images.shape[0] # of imgs in batch
        
    # Final Mean (sum of means * number of samples / total number of samples)
    mean = channel_sum / channel_count
    return mean.tolist()

def calculate_std(mean, loader):
    mean_tensor = torch.tensor(mean).view(1, 3, 1, 1)
    
    # Initialize running sum of squared differences
    channel_std_sum = torch.zeros(3)
    channel_count = 0
    
    for images, _ in loader:
        # Calculate the difference squared: (Image - Mean)^2
        # This uses broadcasting: (B, 3, H, W) - (1, 3, 1, 1)
        diff = (images - mean_tensor).pow(2)
        
        # Sum the squared differences across H, W, and B (Batch)
        channel_std_sum += torch.sum(diff, dim=[0, 2, 3])
        channel_count += diff.shape[0] * diff.shape[2] * diff.shape[3]
        
    # The final Standard Deviation is the square root of the average squared difference (Variance)
    # We use (N - 1) for the denominator for sample standard deviation, but N (total pixels) 
    # is often used for large datasets to simplify.
    std = torch.sqrt(channel_std_sum / channel_count) 
    return std.tolist()

In [5]:
# Calculate mean and std from your dataset
dataset_mean = calculate_mean(temp_loader)
dataset_std = calculate_std(dataset_mean, temp_loader)

print(f"Dataset Mean: {dataset_mean}")
print(f"Dataset Std: {dataset_std}")

# Create the final transform WITH standardization
final_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=dataset_mean, std=dataset_std)  # This actually standardizes
])

# Create final dataset with standardization
dataset = datasets.ImageFolder('./data', transform=final_transform)

# Split into train/val/test
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size]
)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

print(f"\nTrain: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Dataset Mean: [0.4876047968864441, 0.4831056296825409, 0.4614783823490143]
Dataset Std: [0.2261897623538971, 0.22046835720539093, 0.23990651965141296]

Train: 476, Val: 102, Test: 103
